In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

z = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 3))
plt.plot(z, sigmoid(z), linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', label='threshold = 0.5')
plt.xlabel('z'); plt.ylabel('sigmoid(z)')
plt.title('Sigmoid Function'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, epochs=1000, l2=0.0):
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.W = None
        self.b = None
        self.losses = []

    def fit(self, X, y):
        n, d = X.shape
        self.W = np.zeros(d)
        self.b = 0.0
        self.losses = []

        for _ in range(self.epochs):
            z = X @ self.W + self.b
            y_hat = sigmoid(z)
            self.losses.append(binary_cross_entropy(y, y_hat))

            dW = (X.T @ (y_hat - y)) / n + self.l2 * self.W
            db = np.mean(y_hat - y)
            self.W -= self.lr * dW
            self.b -= self.lr * db

        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.W + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

print('LogisticRegression class defined.')

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Breast Cancer dataset:', X.shape)
print('Classes:', cancer.target_names)
print('Train/Test split:', X_train.shape[0], '/', X_test.shape[0])

In [ ]:
model = LogisticRegression(lr=0.1, epochs=1000)
model.fit(X_train, y_train)

print(f'Train Accuracy: {model.accuracy(X_train, y_train):.4f}')
print(f'Test  Accuracy: {model.accuracy(X_test,  y_test):.4f}')
print(f'Final Loss:     {model.losses[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(model.losses)
plt.title('Logistic Regression - Training Loss (Binary Cross-Entropy)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.tight_layout(); plt.show()

In [ ]:
preds = model.predict(X_test)
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0, 1], cancer.target_names, rotation=15)
plt.yticks([0, 1], cancer.target_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix - Breast Cancer')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=14,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

print(classification_report(y_test, preds, target_names=cancer.target_names))

In [ ]:
X2d, y2d = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    random_state=42, n_clusters_per_class=1
)
X2d = StandardScaler().fit_transform(X2d)

model2d = LogisticRegression(lr=0.5, epochs=2000)
model2d.fit(X2d, y2d)
print(f'2D dataset accuracy: {model2d.accuracy(X2d, y2d):.4f}')

x_min, x_max = X2d[:, 0].min() - 0.5, X2d[:, 0].max() + 0.5
y_min, y_max = X2d[:, 1].min() - 0.5, X2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = model2d.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
plt.scatter(X2d[:, 0], X2d[:, 1], c=y2d, cmap='RdBu', edgecolors='k', s=40)
plt.title('Logistic Regression - Decision Boundary')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.tight_layout(); plt.show()

In [ ]:
print(f'Effect of L2 regularisation on breast cancer dataset:')
print(f'{"Lambda":>10} | {"Train Acc":>10} | {"Test Acc":>10}')
print('-' * 36)
for l2 in [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]:
    m = LogisticRegression(lr=0.1, epochs=1000, l2=l2)
    m.fit(X_train, y_train)
    print(f'{l2:>10.3f} | {m.accuracy(X_train, y_train):>10.4f} | {m.accuracy(X_test, y_test):>10.4f}')

In [ ]:
sk_model = SklearnLR(max_iter=1000)
sk_model.fit(X_train, y_train)

print(f'sklearn LogisticRegression Test Accuracy: {sk_model.score(X_test, y_test):.4f}')
print(f'From-scratch Test Accuracy:               {model.accuracy(X_test, y_test):.4f}')